# Three ways to keep earlier tokens in NumPy

A decoder that generates text one token at a time has to remember what came
before. Three methods in the chapter do it three ways: a cache that grows with
the text, a cache that keeps only the last few tokens, and a running state of
fixed size that is added to rather than stored. You will write all three loops
in NumPy over the chapter's own model, then check that each one agrees with the
calculation it is supposed to replace.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. The model is the running example of the chapter: 2 heads of
2 numbers, and a token vector of 4 numbers built from an embedding and its
position values. Head 1 keeps the second and third numbers of the token vector
as its key and the first and second as its value, so the tables below are the
ones the pages print.

In [ ]:
import numpy as np

VOCAB = ["<end>", "the", "cat", "sat", "on", "mat", "dog", "ran"]

E = np.array([[0, 0, 0, -1], [1, 0, 0, 0], [0, 1, 1, 0], [0, 0, 1, 1],
              [-1, -1, 0, 1], [1, 1, 0, 0], [0, 1, 0, 1], [1, 0, 1, 0]], dtype=float)
W_Q = np.zeros((4, 4))
W_Q[2, 0] = W_Q[3, 1] = W_Q[1, 2] = W_Q[3, 3] = 1
W_K = np.zeros((4, 4))
W_K[1, 0] = W_K[2, 1] = W_K[1, 2] = W_K[3, 3] = 1
W_V = np.eye(4)


def positions(T, d=4):
    """Token x_t sits at position p = t."""
    P = np.zeros((T, d))
    for p in range(T):
        for i in range(d // 2):
            w = 1 / 10000 ** (2 * i / d)
            P[p, 2 * i], P[p, 2 * i + 1] = np.sin(w * p), np.cos(w * p)
    return P


def head_one(ids):
    """The queries, keys and values of head 1 for a text, three arrays of shape (T, 2)."""
    H = E[np.asarray(ids)] + positions(len(ids))
    Q, K, V = H @ W_Q, H @ W_K, H @ W_V
    return Q[:, :2], K[:, :2], V[:, :2]


def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()


def attention(q, K, V):
    """One query against the keys it is given, scores divided by sqrt(2)."""
    return softmax(np.asarray(K) @ q / np.sqrt(q.size)) @ np.asarray(V)


def positive(x):
    """The chapter's feature map: 1 added above zero, e to the x below zero."""
    x = np.asarray(x, dtype=float)
    return np.where(x > 0, x + 1.0, np.exp(np.minimum(x, 0.0)))


def linear_attention(q, K, V, gamma=1.0):
    """One query against the keys it is given, without a softmax. A token a steps
    back counts gamma to the power a."""
    sims = positive(K) @ positive(q)
    weights = gamma ** np.arange(len(K) - 1, -1, -1) * sims
    return (weights / weights.sum()) @ np.asarray(V)


def causal_attention(Q, K, V, w=None):
    """Every token at once, under a causal mask and an optional window of w
    tokens ending at the token itself."""
    rows = []
    for t in range(len(Q)):
        first = 0 if w is None else max(0, t - w + 1)
        rows.append(attention(Q[t], K[first:t + 1], V[first:t + 1]))
    return np.array(rows)


def direct_linear(Q, K, V, gamma=1.0):
    """Every token at once, with the similarities of the keys that came before."""
    return np.array([linear_attention(Q[t], K[:t + 1], V[:t + 1], gamma)
                     for t in range(len(Q))])


PROMPT = [1, 2, 3]
LONGER = [1, 2, 3, 4]
print(" ".join(VOCAB[i] for i in LONGER))

## Part 1 (coding task): Decode with a cache of keys and values

Write `decode_with_cache(Q, K, V)`. The position of each token is already
inside `Q`, `K` and `V` (the setup's `positions` gives token `t` position
`t`). At step `t`:

1. Add `K[t]` and `V[t]` to the cache, so the token's own key is in it.
2. Score `Q[t]` against every key in the cache with the `attention(q, K, V)`
   helper from the setup, since the check counts how many keys each call is
   given.

Return the `T` outputs as one array of shape `(T, 2)`, and leave `Q`, `K` and
`V` as they were.

In [ ]:
def decode_with_cache(Q, K, V):
    """Causal attention one token at a time, keeping every key and value."""
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = head_one(LONGER)
n = len(Q)
given = (Q.copy(), K.copy(), V.copy())

scored = []
score_step = attention


def attention(q, K_, V_):        # noqa: F811, the check counts the keys it is given
    scored.append(len(K_))
    return score_step(q, K_, V_)


try:
    out = np.asarray(decode_with_cache(Q, K, V))
finally:
    attention = score_step

assert out.shape == (n, 2), f"decode_with_cache returned shape {out.shape}, expected ({n}, 2)"
assert all(np.array_equal(a, b) for a, b in zip((Q, K, V), given)), "decode_with_cache changed Q, K or V in place"
want = causal_attention(Q, K, V)
if not np.allclose(out, want):
    if sum(scored) == n * (n - 1) // 2:
        raise AssertionError("every token's own key was left out of its own row: add the new key to the cache before scoring the new query")
    if sum(scored) == n * n:
        raise AssertionError("every call was given all n keys: a decode run scores only the keys in the cache, which grows by one each step")
    first = int(np.argmax(~np.isclose(out, want).all(axis=1)))
    raise AssertionError(f"row {first} is not what causal attention gives")
assert sum(scored) == n * (n + 1) // 2, f"the calls were given {sum(scored)} keys in all, so {n} calls saw {sum(scored) // n} each: a decode run on token t sees t + 1 keys"
assert np.allclose(out[3], [0.722, 0.373], atol=0.002), f"on 'the cat sat on' head 1 of the chapter returns (0.722, 0.373), got {np.round(out[3], 3)}"
print("Part 1 passed")

## Part 2 (coding task): A rolling cache of w slots

Write `window_decode(Q, K, V, w)`. The cache is a circular buffer of
`w` slots, and token `t` goes into slot `t % w`, overwriting the token `w`
places before it. Score `Q[t]` against the keys now in the cache, which are the
last `w` tokens, or all of them if fewer have been read. Use the
`attention(q, K, V)` helper, and read the slots in position order.

Return the outputs as `(T, 2)` and the cache of keys as `(w, 2)`, in slot
order, so the check can see which slot each token landed in.

In [ ]:
def window_decode(Q, K, V, w):
    """Causal attention one token at a time, with a window of w tokens."""
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = head_one(LONGER)
n, w = len(Q), 3
out, cache = window_decode(Q, K, V, w)
out, cache = np.asarray(out), np.asarray(cache)

assert out.shape == (n, 2), f"window_decode returned outputs of shape {out.shape}, expected ({n}, 2)"
assert cache.shape == (w, 2), f"the cache came back with shape {cache.shape}, expected ({w}, 2): a window of w tokens keeps w slots however long the text is"
assert np.allclose(out[0], V[0]), f"the first token sees only itself, so it returns its own value {V[0]}, got {np.round(out[0], 3)}"
want = causal_attention(Q, K, V, w)
if not np.allclose(out, want):
    if np.allclose(out, causal_attention(Q, K, V, w + 1)):
        raise AssertionError(f"the window is one token too wide: token t may score tokens t - {w} + 1 to t, so {w} keys in all")
    if np.allclose(out, causal_attention(Q, K, V)):
        raise AssertionError(f"every row was computed as if there were no window: a token of position t may score only the {w} tokens ending at t")
    first = int(np.argmax(~np.isclose(out, want).all(axis=1)))
    raise AssertionError(f"row {first} is not what attention over a window of {w} gives")

lands = np.array([K[t] for t in range(n)])
slots = [n - 1 - (n - 1 - s) % w for s in range(w)]
assert np.allclose(cache, lands[slots]), f"after {n} tokens a window of {w} holds the keys of the last {w} tokens, with token t in slot t % {w}: slots {slots}, got {np.round(cache, 3).tolist()}"
print("Part 2 passed")

## Part 3 (coding task): A running state with a decay

Write `linear_decode(Q, K, V, gamma=1.0)`. Keep a state of shape `(2, 2)`, one
total for every pair of a key number and a value number, and a normalizer of
shape `(2,)` holding the total of every key number. At step `t`:

1. Multiply both by `gamma`.
2. Add `np.outer(positive(K[t]), V[t])` to the state and `positive(K[t])` to the
   normalizer.
3. Return `positive(Q[t]) @ state / (positive(Q[t]) @ normalizer)`.

Return the outputs as `(T, 2)`, then the state and the normalizer.

In [ ]:
def linear_decode(Q, K, V, gamma=1.0):
    """Linear attention one token at a time, from a running state."""
    # TODO
    raise NotImplementedError

In [ ]:
Q, K, V = head_one(PROMPT)
out, state, z = linear_decode(Q, K, V)
out, state, z = np.asarray(out), np.asarray(state), np.asarray(z)
assert out.shape == (3, 2), f"linear_decode returned outputs of shape {out.shape}, expected (3, 2)"
assert state.shape == (2, 2) and z.shape == (2,), f"the state came back with shape {state.shape} and the normalizer with {z.shape}, expected (2, 2) and (2,): the state is one total per pair of key and value numbers"
want = direct_linear(Q, K, V)
if not np.allclose(out, want):
    if np.any(~np.isfinite(out)):
        raise AssertionError("an output is not a number: the token is added to the state before its own output is read, or the normalizer was left at zero")
    first = int(np.argmax(~np.isclose(out, want).all(axis=1)))
    raise AssertionError(f"row {first} is not what linear attention gives")
assert np.allclose(out[2], [0.905, 0.827], atol=0.002), f"on 'the cat sat' linear attention returns (0.905, 0.827), got {np.round(out[2], 3)}"

Q, K, V = head_one(PROMPT)
out_half, state_half, z_half = linear_decode(Q, K, V, 0.5)
out_half = np.asarray(out_half)
if not np.allclose(out_half, direct_linear(Q, K, V, 0.5)):
    raise AssertionError("with a decay of 0.5 the rows do not match: the normalizer is a sum of key numbers, so it has to be multiplied by gamma as well as the state")
assert np.allclose(out_half[2], [0.894, 0.494], atol=0.002), f"with a decay of 0.5 the answer for 'sat' is (0.894, 0.494), got {np.round(out_half[2], 3)}"

Q, K, V = head_one([1, 2, 3, 4, 1, 2, 3, 4])
_, state_long, z_long = linear_decode(Q, K, V)
assert state_long.shape == (2, 2) and z_long.shape == (2,), "the state grew with the text: it is 4 numbers and a 2-number normalizer after 3 tokens or after 3,000"
print("Part 3 passed")

## Part 4 (coding task): Run a text three ways and count what each one keeps

Write `run_all(ids, w=3, gamma=1.0)`, which runs the text through the three
functions above and returns a dictionary with

- `"cached"`, the outputs of `decode_with_cache`, shape `(T, 2)`
- `"windowed"`, the outputs of `window_decode` with the window `w`
- `"linear"`, the outputs of `linear_decode` with the decay `gamma`
- `"numbers"`, how many numbers each mode is holding once the text is read, as
  `(cached, windowed, linear)`

Use `head_one(ids)` to get the queries, keys and values of head 1.

In [ ]:
def run_all(ids, w=3, gamma=1.0):
    """Run a text with a full cache, a window and a running state."""
    # TODO
    raise NotImplementedError

In [ ]:
TEXT = [1, 2, 3, 4, 1, 2]
w, gamma = 3, 0.5
Q, K, V = head_one(TEXT)
n = len(Q)
got = run_all(TEXT, w=w, gamma=gamma)

for key in ("cached", "windowed", "linear"):
    assert np.shape(got[key]) == (n, 2), f"run_all's {key} has shape {np.shape(got[key])}, expected ({n}, 2)"
cached, windowed, linear = (np.asarray(got[key]) for key in ("cached", "windowed", "linear"))
assert np.allclose(cached, causal_attention(Q, K, V)), "the cached outputs should be ordinary causal attention over every key"
assert np.allclose(windowed, causal_attention(Q, K, V, w)), f"the windowed outputs should be attention over the last {w} keys of each token"
assert np.allclose(linear, direct_linear(Q, K, V, gamma)), f"the linear outputs should be linear attention with a decay of {gamma}"
assert np.allclose(cached[0], windowed[0]), "the first token sees only itself, so a window of any width cannot change its output"
assert not np.allclose(cached[2:], windowed[2:]), f"with a window of {w} the later tokens look no further back than {w - 1} places, so some output has to change"
assert not np.allclose(cached[1:], linear[1:]), "linear attention has no softmax, so it should differ from the softmax on the second token already"
numbers = np.asarray(got["numbers"], dtype=float)
want_numbers = [2 * n * 2, 2 * w * 2, 2 * 2 + 2]
assert numbers.shape == (3,) and np.allclose(numbers, want_numbers), f"after {n} tokens the modes hold {want_numbers[0]}, {want_numbers[1]} and {want_numbers[2]} numbers, got {np.asarray(got['numbers']).tolist()}"
print("Part 4 passed")

## Solution

### Part 1: Decode with a cache of keys and values

In [ ]:
def decode_with_cache(Q, K, V):
    """Causal attention one token at a time, keeping every key and value."""
    keys, values, out = [], [], []
    for t in range(len(Q)):
        keys.append(K[t])
        values.append(V[t])
        out.append(attention(Q[t], np.array(keys), np.array(values)))
    return np.array(out)

### Part 2: A rolling cache of w slots

In [ ]:
def window_decode(Q, K, V, w):
    """Causal attention one token at a time, with a window of w tokens."""
    cache = np.zeros((w, V.shape[1]))
    out = []
    for t in range(len(Q)):
        cache[t % w] = K[t]
        first, filled = max(0, t - w + 1), min(t + 1, w)
        order = [(first + j) % w for j in range(filled)]
        out.append(attention(Q[t], cache[order], V[first:t + 1]))
    return np.array(out), cache

### Part 3: A running state with a decay

In [ ]:
def linear_decode(Q, K, V, gamma=1.0):
    """Linear attention one token at a time, from a running state."""
    state, normalizer, out = np.zeros((V.shape[1], V.shape[1])), np.zeros(V.shape[1]), []
    for t in range(len(Q)):
        key = positive(K[t])
        state, normalizer = gamma * state + np.outer(key, V[t]), gamma * normalizer + key
        query = positive(Q[t])
        out.append(query @ state / (query @ normalizer))
    return np.array(out), state, normalizer

### Part 4: Run a text three ways and count what each one keeps

In [ ]:
def run_all(ids, w=3, gamma=1.0):
    """Run a text with a full cache, a window and a running state."""
    Q, K, V = head_one(ids)
    n = len(Q)
    cached = decode_with_cache(Q, K, V)
    windowed, cache = window_decode(Q, K, V, w)
    linear, state, z = linear_decode(Q, K, V, gamma)
    return {"cached": cached, "windowed": windowed, "linear": linear,
            "numbers": (2 * n * V.shape[1], 2 * w * V.shape[1],
                        state.size + z.size)}

**Why it works.** The causal mask means the rows of the score table for earlier
tokens never change when a token is appended, so the cache holds exactly the
numbers a run from scratch would have recomputed and the outputs are identical.
What changes is the count: the $t$-th decode run scores $t + 1$ keys instead of
$t + 1$ rows of the table, so writing $n$ tokens costs $n(n+1)/2$ scores rather
than the sum of the squares. Queries are not cached. They are used once, by
their own token, and no later token needs them, so keeping them would spend
memory on numbers nobody reads.

**Why it works.** A key older than the window is never read again, since no
query after it will look that far back, so it can be overwritten. Reading the
slots in position order is what keeps the window contiguous: the $w$ keys of the
current window sit at slots $(t - w + 1) \bmod w$ through $t \bmod w$, which
wraps around the end of the buffer exactly once. The values are read by
position alongside the keys, so the caller needs the cache of keys only to check
the layout. Skipping the keys outside the window does not make attention exact
any more, which is the difference from FlashAttention, and it is also what keeps
the work per token the same at any length.

**Why it works.** The query appears in every term of the linear attention sum,
so it can be taken out and the rest collected into a state that does not depend
on it. The state is the total of $\phi(k) v^\top$ over the tokens read so far and
the normalizer is the total of $\phi(k)$, so
$\dfrac{\phi(q) \cdot S}{\phi(q) \cdot z}$ is the same weighted sum the definition
gives, with the weights written as similarities over their total. Nothing is
stored per token, which is why the state is 6 numbers after 3 tokens and after
3,000. Multiplying the normalizer by $\gamma$ as well is what makes a token $a$
steps back count $\gamma^a$: it is a sum of the same kind, so it decays like one.

**Why it works.** The three modes read the same text and the same weights, so
every difference between them comes from what each one lets a token see. The
first token is the one case where all three must agree, since one visible key
gives weight 1 whatever the method. While the window still holds every
earlier token, the cache and the window agree as well. After that the window
drops keys the causal mask would have allowed, and linear attention keeps all
of them but replaces the softmax, so its weights are flatter even where
nothing was dropped.

The counts are the price of each choice. A full cache holds a key and a value
per token per head, so it grows with every token generated. A window holds $w$
of them and never grows, at the cost of what a token can reach directly. The
running state holds $d^2 + d$ numbers whatever the length, which is why memory
stays fixed during generation. Only the window and the state change what the
model computes, and the cache changes nothing but the work.